# Genie API Client Demo

**Author:** Sean Zhang  
**Version:** v0.2  
**Date:** Feb 2026

---

## Overview
This demo notebook showcases the capabilities of the Genie API Client package, demonstrating robust strategies for interacting with the Databricks Genie API. It focuses on key concepts such as exponential backoff, polling, and stress testing, which are essential for managing API rate limits and ensuring efficient communication with asynchronous services.

### Authentication
This client uses the **Databricks SDK `WorkspaceClient`** for authentication.
- **In Databricks notebooks:** Workspace-level auth is automatic.
- **Locally:** Uses `~/.databrickscfg` profiles or other [supported auth methods](https://databricks-sdk-py.readthedocs.io/en/latest/authentication.html).

### Purpose and Context
The primary goal of this demo is to provide practical examples of how to use the modularized Genie API client components. It demonstrates handling common challenges when working with APIs, particularly in scenarios where rate limits (HTTP 429 errors) may occur. By employing exponential backoff with jitter, users can minimize the risk of overwhelming the API and improve the chances of successful requests.

The Genie API has rate limits that may be encountered under load. The [Genie API documentation](https://docs.databricks.com/aws/en/genie/conversation-api) outlines best practices, and this repo provides ready-to-use templates for implementing them.

### A Note on SDK Retry Behavior
The Databricks SDK's `_api.do()` includes built-in retry handling for 429 responses, automatically waiting based on the server's `Retry-After` header (~60s). This is convenient for many workflows, but means retry behavior is managed at the SDK layer rather than the application layer.

This demo uses `GenieClient` (SDK auth + raw `requests`) for **application-level rate limit visibility and custom backoff**. If you prefer SDK-managed retries, see `GenieClientSDK` in `genie_client_sdk.py` and set `debug=True` to observe retry events in logs.

### Demo Structure
The notebook demonstrates several key capabilities:
- **Package Imports:** How to import and use the modularized components
- **Client Configuration:** Setting up clients with different timing configurations
- **Stress Testing:** Running concurrent tests to evaluate rate limit handling
- **Trace Analysis:** Examining detailed logs and metrics from API interactions

By following the examples in this demo, you can begin implementing Genie API best practices into your own Genie and Multi-Agent workflows.

## Key Features Demonstrated
- **SDK Authentication:** Uses `WorkspaceClient` for seamless auth
- **Exponential Backoff:** Retry strategy with exponential backoff and jitter when encountering rate limits
- **Configurable Polling:** Polls at regular intervals until completion or timeout with configurable parameters
- **MLflow Integration:** All Genie API interactions are traced and logged for analysis and reproducibility
- **Stress Testing:** Simulates high-load scenarios by sending multiple questions concurrently
- **Multiple Configurations:** Shows different timing configurations for various use cases

## Package Components Used
- **genie_client.py:** `GenieClient` -- SDK auth + raw requests (application-level 429 visibility and custom backoff)
- **genie_client_sdk.py:** `GenieClientSDK` -- pure SDK version (SDK manages retries automatically; use `debug=True` to observe retry events)
- **stress_test.py:** Utilities for concurrent testing and benchmarking
- **config.py:** Predefined timing configurations and test parameters

---

**This demo showcases a starting point for building resilience into Genie API calls.**

## Import Package Components

This section demonstrates how to import the GenieClient package components that we'll use throughout this demo. The GenieClient handles all the complexity of API communication, rate limiting, and error handling behind a simple interface. We'll also import various timing configurations that demonstrate different polling and backoff strategies.

In [0]:
%pip install -r requirements.txt
dbutils.library.restartPython()

In [0]:
# Import the GenieClient and related utilities from our package
from genie_client import GenieClient
from config import (
    DEFAULT_TIMING_CONFIG, 
    TIMING_CONFIG_EARLY_BACKOFF, 
    TIMING_CONFIG_EARLY_TIMEOUT,
    SPACE_ID
)

print("GenieClient and configuration imported successfully!")
print("Available timing configurations:")
print("  DEFAULT_TIMING_CONFIG: Standard production settings") 
print("  TIMING_CONFIG_EARLY_BACKOFF: Faster backoff for testing")
print("  TIMING_CONFIG_EARLY_TIMEOUT: Short timeouts for testing")
print(f"\nAuthentication: Handled by Databricks SDK WorkspaceClient (automatic)")
print(f"Space ID: {SPACE_ID}")

## Create Client Instances

This section demonstrates how to create multiple GenieClient instances with different timing configurations. Each client uses the same `WorkspaceClient` for authentication (handled automatically by the Databricks SDK) but employs different strategies for handling rate limits, polling intervals, and timeouts. This allows us to compare how different configurations behave under various conditions.

In [0]:
# Authentication is handled automatically by the Databricks SDK WorkspaceClient.
# In Databricks notebooks: workspace auth is automatic (no setup needed).
# Locally: uses ~/.databrickscfg profiles or other supported auth methods.
# See: https://databricks-sdk-py.readthedocs.io/en/latest/authentication.html

from databricks.sdk import WorkspaceClient

# Two client implementations are available:
#   GenieClient     - SDK auth + raw requests (application-level 429 visibility, custom backoff)
#   GenieClientSDK  - Pure SDK _api.do (SDK manages retries using server Retry-After; use debug=True for visibility)
#
# This demo uses GenieClient for full observability of rate limiting behavior.
# See genie_client_sdk.py if you prefer SDK-managed retries.

# Create a shared WorkspaceClient (optional - GenieClient creates one if not provided)
w = WorkspaceClient()

print("Setting up GenieClient instances...")
print(f"Space ID: {SPACE_ID}")
print(f"Auth: {w.config.auth_type}")

# Standard client with default timing configuration
default_client = GenieClient(
    space_id=SPACE_ID,
    client=w
)
print("Standard GenieClient created (SDK auth + raw requests)")

# Client with early backoff for testing
early_backoff_client = GenieClient(
    space_id=SPACE_ID,
    client=w,
    timing_config=TIMING_CONFIG_EARLY_BACKOFF
)
print("Early backoff GenieClient created")

# Client with early timeout for testing
early_timeout_client = GenieClient(
    space_id=SPACE_ID,
    client=w,
    timing_config=TIMING_CONFIG_EARLY_TIMEOUT
)
print("Early timeout GenieClient created")

print("\nReady to demonstrate Genie API capabilities!")

## Import Stress Testing Utilities

This section imports the stress testing and benchmarking utilities from our package. These tools allow us to evaluate API performance under load, test different timing configurations, and analyze the effectiveness of our retry and backoff strategies. The utilities provide both individual stress testing functions and comprehensive benchmarking capabilities.

In [0]:
# Import stress testing utilities
from stress_test import stress_test_api_limit
from config import DEFAULT_STRESS_TEST_PARAMS

print("Stress testing utilities imported!")
print("Available functions:")
print("   - stress_test_api_limit(): Run concurrent API stress tests")
print("   - analyze_results(): Analyze and summarize test results")
print("\nTest configurations available:")
print(f"   - Default stress test: {DEFAULT_STRESS_TEST_PARAMS}")

### Default Configuration Stress Test

This section demonstrates the default client's behavior under load by sending multiple concurrent questions to the Genie API. This intentionally triggers rate limits to showcase:
- How custom exponential backoff with jitter handles high load
- The client's response to rate limiting (HTTP 429 errors) -- visible in trace logs  
- Collection of detailed metrics on response times, error rates, and system resilience

> **Note:** This demo uses `GenieClient` (raw `requests`) so that 429 events appear directly in trace data. If using `GenieClientSDK`, retries are managed by the SDK; set `debug=True` to observe them in logs.

The results and trace data provide insights into API performance and client behavior patterns.

In [0]:
# Run stress test with standard configuration
print(f"Starting stress test with {DEFAULT_STRESS_TEST_PARAMS["num_questions"]} questions over {DEFAULT_STRESS_TEST_PARAMS["time_frame_s"]} seconds...")
print("WARNING: This will intentionally hit rate limits to test backoff behavior")

default_results = stress_test_api_limit(
    genie_client=default_client, 
    question_param = "cancer type", # The default stress test question is "What is the top {question_param}?"
    num_questions=DEFAULT_STRESS_TEST_PARAMS["num_questions"], 
    time_frame_s=DEFAULT_STRESS_TEST_PARAMS["time_frame_s"]
)

print(f"Stress test completed! Generated {len(default_results)} results")
print("Check the results below and trace data in subsequent cells")

In [0]:
# Default stress test results
display(default_results)

In [0]:
# Default client trace results (Filter on a particular question_id and order by timestamp to see full trace)
display(default_client.get_trace_df())

### Early Backoff Test

This section demonstrates how the GenieClient adapts its polling strategy when responses are delayed. By configuring a short `poll_backoff_after` value, we can observe the transition from regular polling to exponential backoff. This shows how the client automatically adjusts to API conditions and prevents overwhelming slow or congested endpoints.

In [0]:
# Test early backoff behavior (backoff starts after 5 seconds)
print("Testing early backoff configuration...")
print("NOTE: This client switches to exponential backoff after just 5 seconds of polling")

early_backoff_results = stress_test_api_limit(early_backoff_client, "cancer type", num_questions=1, time_frame_s=1)

print("Early backoff test completed!")
print("TIP: Check timing_config below to see the early backoff settings")

In [0]:
early_backoff_client.timing_config

### Early Timeout Test

This section demonstrates the GenieClient's timeout enforcement capabilities. By configuring a short `max_poll_wait` value, we can observe how the client handles situations where API responses take longer than expected. This test shows the client's resilience and demonstrates how timeout configurations prevent indefinite waits in production environments.

In [0]:
# Test early timeout behavior (times out after 5 seconds)
print("Testing early timeout configuration...")  
print("NOTE: This client will timeout after just 5 seconds instead of the default 10 minutes")

early_timeout_results = stress_test_api_limit(early_timeout_client, "cancer type", num_questions=1, time_frame_s=1)

print("Early timeout test completed!")
print("Check results and trace data below to see timeout behavior")

In [0]:
display(early_timeout_results)

In [0]:
display(early_timeout_client.get_trace_df())

## Demo Summary & Next Steps

This concludes our demo of the Genie API Client package. We've demonstrated the key capabilities including exponential backoff, configurable polling, stress testing, and trace analysis.

### Key Takeaway: Choosing Your Retry Strategy
This repo provides two client implementations to match different needs. `GenieClient` (SDK auth + raw `requests`) gives you application-level visibility into rate limiting and custom backoff strategies. `GenieClientSDK` delegates retry handling to the SDK for a simpler integration -- set `debug=True` to observe retry events in logs.

### Next Steps
- Integrate these components into your own Genie and Multi-Agent workflows
- Tune timing configurations based on your workload patterns
- Use trace data to monitor and optimize API interactions
